# 01: EDA (team 04)

Covers what the Deliverable 3 brief asks for: shape and types, missingness, distributions and
outliers, leakage risk, and relationships that bear on model choice.

Every section adds to `findings`, an observation and its modelling consequence. The numbers and the
wording are computed when the notebook runs, so no result is written into this file. Section 7
collects them. The first line of output says whether this is fake or real data. On fake data the
findings only prove that the code can detect the quirks planted in `make_fake_data.py`.

> **NDA.** This notebook prints aggregates only, never rows or airport codes. Clear outputs before committing.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config as c
from features import airport_attributes, add_lag, clean, miles_between

raw, airports = c.load()
pax = clean(raw)
route = pax[c.ROUTE_KEYS[0]] + "|" + pax[c.ROUTE_KEYS[1]]
y = pax[c.TARGET]
findings = []


def note(section, observation, consequence):
    findings.append({"section": section, "observation": observation, "consequence": consequence})


def save(name):
    plt.tight_layout()
    plt.savefig(c.RESULTS_DIR / f"{name}.png", dpi=120)
    plt.show()

## 1. Shape, types, and what one row is

In [ ]:
for name, table in [("pax", raw), ("airports", airports)]:
    print(f"{name}: {len(table):,} rows, {table.shape[1]} columns")
    display(pd.DataFrame({"dtype": table.dtypes.astype(str), "missing_%": (table.isna().mean() * 100).round(2), "unique": table.nunique()}))

In [ ]:
dupes = len(raw) - len(raw.drop_duplicates())
grain = c.ROUTE_KEYS + [c.DATE]
grain_repeats = int(pax.duplicated(grain).sum())
self_loops = int((pax[c.ROUTE_KEYS[0]] == pax[c.ROUTE_KEYS[1]]).sum())
fractional = (y % 1 != 0).mean()
print(f"exact duplicates {dupes:,}, grain repeats {grain_repeats:,}, same origin and destination {self_loops:,}, non-integer target {fractional:.1%}")

note("1", f"{dupes:,} exact duplicate rows", "Dropped in clean()." if dupes else "Nothing to drop.")
note("1", f"{grain_repeats:,} rows repeat origin + destination + month",
     "Grain confirmed: one directional route per month." if grain_repeats == 0 else "Grain is wrong: find the missing key before modelling.")
note("1", f"{fractional:.0%} of target values aren't whole numbers",
     "These aren't raw counts. Ask a PM what the column measures before reporting it as passengers." if fractional > 0.5 else "Consistent with raw counts.")

## 2. Missingness

In [ ]:
periods = pax[c.DATE].drop_duplicates().sort_values()
total = pax.groupby(c.DATE)[c.TARGET].sum()

print("Cells missing from the pax table:", int(raw.isna().sum().sum()))
airport_gaps = airports.isna().mean().loc[lambda s: s > 0]
print("Airport table gaps:\n" + (airport_gaps * 100).round(2).to_string())

routes_total = route.nunique()
fill = len(pax) / (routes_total * len(periods))
per_route = route.value_counts()
full = (per_route == len(periods)).mean()
volume = y.groupby(route).sum()
coverage_vs_volume = volume.corr(per_route.reindex(volume.index), method="spearman")
print(f"routes {routes_total:,}, route-month cells filled {fill:.1%}, routes with every month {full:.1%}, coverage vs volume rank corr {coverage_vs_volume:.2f}")

note("2", f"{1 - fill:.1%} of route-months have no row, and {full:.0%} of routes have every month",
     "Gaps are absent rows, not blanks. Lags match on the actual date, because shift(1) would pick the wrong month.")
note("2", f"Coverage vs route volume rank correlation {coverage_vs_volume:.2f}",
     "Thinner routes skip more, so absence isn't random. A missing month may mean zero or below a threshold: ask a PM before filling." if coverage_vs_volume > 0.2 else "Absence isn't clearly tied to volume.")
if len(airport_gaps):
    note("2", f"{len(airport_gaps)} airport columns have gaps", "Check whether each gap is tied to a group of airports before using that column.")

## 3. Target distribution, outliers, and any shock

In [ ]:
print(y.describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]).round(1).to_string())
skew = y.skew()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(y, bins=60)
axes[0].set_title("target")
axes[1].hist(np.log1p(y), bins=60)
axes[1].set_title("log1p(target)")
save("target_distribution")

top = volume.sort_values(ascending=False)
top_share = top.head(len(top) // 10).sum() / top.sum()
print(f"skew {skew:.1f}; top 10% of routes carry {top_share:.1%} of the target")

note("3", f"Target skew {skew:.1f}", "Model log1p(target), so big routes don't dominate the loss." if skew > 1 else "Skew is mild: the raw target is fine.")
note("3", f"Top 10% of routes carry {top_share:.0%} of the target",
     "One score hides this. Report volume-weighted WAPE for the network and median route error for thin routes.")

In [ ]:
shock = total < 0.7 * total.median()
shock_months = shock[shock].index
shock_rows = pax[c.DATE].isin(shock_months)
has_shock = len(shock_months) > 0
print(f"months over 30% below the median total: {len(shock_months)}" + (f" ({shock_months.min():%Y-%m} to {shock_months.max():%Y-%m})" if has_shock else ""))

route_median = y[~shock_rows].groupby(route[~shock_rows]).median()
ratio = y / route.map(route_median)
outliers = ((ratio > 5) | (ratio < 0.2)) & ~shock_rows
print(f"rows outside any shock over 5x or under 1/5 of their route median: {outliers.sum():,} ({outliers.mean():.2%})")

fig, ax1 = plt.subplots(figsize=(10, 3))
ax1.plot(total / total.median())
ax1.set_title("Total target per month, relative to the median month")
save("target_over_time")

yoy = total.iloc[-6:].sum() / total.iloc[-18:-12].sum()
print(f"last 6 months vs the same months a year earlier: {yoy:.3f}")

if has_shock:
    note("3", f"{len(shock_months)} months between {shock_months.min():%Y-%m} and {shock_months.max():%Y-%m} sit far below normal",
         "Test training with and without this window. Validate only on later data, so the score isn't about the shock.")
note("3", f"{outliers.mean():.2%} of rows are far from their route's median",
     "Use absolute-error loss, which is robust, and keep the rows until someone explains them.")
note("3", f"Latest 6 months vs a year earlier: ratio {yoy:.2f}",
     "The level has moved, and trees can't extrapolate from the date, so test the year-over-year feature. Ask a PM if the latest months are final." if abs(yoy - 1) > 0.05 else "No big recent level change.")

## 4. Leakage risk

The test for every column is: *would we know this value at the moment we make the forecast?*

In [ ]:
rev = pax[c.LEAKY[0]]
rank_corr = rev.corr(y, method="spearman")
per_pax = (rev / y).replace([np.inf, -np.inf], np.nan)
print(f"Revenue vs target: rank corr {rank_corr:.3f}; zero revenue rows {(rev == 0).mean():.2%}")
print("Revenue per unit of target:\n" + per_pax.describe(percentiles=[0.01, 0.5, 0.99]).round(1).to_string())

note("4", f"Revenue has rank correlation {rank_corr:.2f} with the target",
     "Leak: revenue is known only after people have flown. Dropped. 02 measures what it would have been worth.")
note("4", "Lags of the target are the obvious candidates, and their legality depends on how far ahead we forecast",
     "HORIZON in config.py enforces this: no lag shorter than the horizon. Ask a PM for the real horizon.")

## 5. Is there a real time axis?

In [ ]:
print(f"{periods.min():%Y-%m} to {periods.max():%Y-%m}: {len(periods)} months, {routes_total:,} routes")
steady = pax.loc[~shock_rows, c.ROUTE_KEYS + [c.DATE, c.TARGET]].copy()
steady[c.TARGET] = np.log1p(steady[c.TARGET])
autocorr = {}
for k in (1, 2, 3, 12):
    lagged = add_lag(steady, k)
    # within-route, so this measures the time signal and not just that big routes stay big
    means = lagged.groupby(c.ROUTE_KEYS)[[c.TARGET, f"LAG_{k}"]].transform("mean")
    autocorr[k] = (lagged[c.TARGET] - means[c.TARGET]).corr(lagged[f"LAG_{k}"] - means[f"LAG_{k}"])
print("Within-route autocorrelation outside any shock:\n" + pd.Series(autocorr).round(3).to_string())

deviation = steady[c.TARGET] - steady.groupby(c.ROUTE_KEYS)[c.TARGET].transform("mean")
season = deviation.groupby(steady[c.DATE].dt.month).mean()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(season.index, season.values)
ax.set_title("Average log deviation from route mean, by calendar month")
save("seasonality")

last_year = periods.iloc[-12:]
seen_before = set(route[pax[c.DATE] < last_year.min()])
new_routes = 1 - route[pax[c.DATE].isin(last_year)].drop_duplicates().isin(seen_before).mean()
print(f"routes in the last 12 months never seen before them: {new_routes:.2%}")

note("5", f"{len(periods)} ordered months across {routes_total:,} routes",
     "A panel over time, so split by date and never randomly. Random splits would train on the future.")
note("5", f"Within-route autocorrelation: lag 1 {autocorr[1]:.2f}, lag 12 {autocorr[12]:.2f}",
     "Last value and same month last year are the baselines to beat.")
note("5", f"Calendar-month effect spans {season.max() - season.min():.2f} in log units",
     "Test month as cyclic features." if season.max() - season.min() > 0.1 else "Seasonality is small: month features are optional.")
note("5", f"{new_routes:.1%} of recent routes are new",
     "Few cold starts, so route history covers nearly every route." if new_routes < 0.01 else "Real cold starts: airport attributes may matter for routes with no history.")
note("5", f"{routes_total:,} separate series, some with gaps",
     "One ARIMA per route would be awkward at this size. A single global model over all routes fits this shape.")

## 6. Relationships and cardinality

In [ ]:
attrs = airport_attributes(airports).set_index(c.AIRPORT_KEY)
org, des = attrs.reindex(pax[c.ROUTE_KEYS[0]]), attrs.reindex(pax[c.ROUTE_KEYS[1]])
miles = pd.Series(miles_between(org["LAT"].values, org["LON"].values, des["LAT"].values, des["LON"].values), index=pax.index)
intl = pd.Series((org["COUNTRY"].values != des["COUNTRY"].values) & org["COUNTRY"].notna().values & des["COUNTRY"].notna().values, index=pax.index)

pearson, spearman = miles.corr(np.log1p(y)), miles.corr(y, method="spearman")
binned = np.log1p(y).groupby(pd.qcut(miles, 10, duplicates="drop"), observed=True).median()
print(f"distance vs log target: pearson {pearson:.2f}, spearman {spearman:.2f}")
print("Median log target by distance decile:\n" + binned.round(2).reset_index(drop=True).to_string())
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(binned) + 1), binned.values, marker="o")
ax.set_title("Median log target by distance decile")
save("distance")

monotone = (binned.diff().dropna() > 0).mean()
note("6", f"Distance vs log target: pearson {pearson:.2f}, and {monotone:.0%} of decile steps go up",
     "Not a straight line: this favours trees, which need no hand-built shapes." if 0.2 < monotone < 0.8 else "Roughly monotone, so a linear term can represent it.")
print(f"\nInternational share of rows: {intl.mean():.1%}")
note("6", f"International routes are {intl.mean():.0%} of rows", "02 tests whether this adds anything beyond the route's own history.")

In [ ]:
cardinality = pax[c.ROUTE_KEYS].nunique()
print(cardinality.to_string())
for key in c.ROUTE_KEYS:
    print(f"{key} found in the airport table: {pax[key].isin(attrs.index).mean():.2%}")
matched = min(pax[key].isin(attrs.index).mean() for key in c.ROUTE_KEYS)

note("6", f"{cardinality.max()} airports and {routes_total:,} routes",
     "Airports fit native categoricals in trees. Routes don't, so route history comes through lags, not a route ID." if cardinality.max() <= 250 else "Too many airports for native categoricals: group the rare ones.")
note("6", f"Airport join matches {matched:.1%} of rows",
     "Safe to join." if matched > 0.99 else "Join is incomplete. Keep a 'missing' category and find out which airports are absent.")

## 7. Findings → modelling consequences

In [ ]:
table = pd.DataFrame(findings)
table.to_csv(c.RESULTS_DIR / "eda_findings.csv", index=False)
pd.set_option("display.max_colwidth", None)
table